# Hook demo (facilitator only)
### Agenda item 1, 00:00-00:08 -- live skit, record, press one button, show minutes

**This notebook is for whoever is running the hook live, not for participants** -- it has the
microphone recorder and the one-button demo. The participants' notebook (`audio_workshop.ipynb`)
does not call this file at all.

Run cell 1 **before the session starts** (it downloads and loads the Whisper model, which takes a
minute or two -- don't do this live in front of the room). Then during the hook: run cell 2 to
record the skit from the browser microphone, cell 3 to press the button on what was just recorded,
and cell 4 if the recording fails and the backup WAV is needed instead.

**The live-skit path (cells 2-3) has not been rehearsed or tested end to end** -- only the backup
WAV (cell 4) has been run and verified. Rehearse the live path before the event (see
`demo/FACILITATOR.md`'s rehearsal checklist); if that hasn't happened yet, use the backup WAV, the
safe route.

**Pacing (if acting it out live):** say each turn in one go, with no long pause inside it (two
turns have two sentences -- say both without stopping); leave a clear ~2-second pause before the
other person speaks; don't overlap. The transcriber splits speakers on a 1.5-second gap, so a
quick, natural back-and-forth gets merged into one speaker, and a long pause inside one turn gets
heard as a speaker change too.

See `demo/FACILITATOR.md` for what to say and when.

## 1. Setup -- run this before the session (preloads Whisper)
On Colab: **Runtime > Change runtime type > T4 GPU** makes this faster. Upload `facilitator_files.zip`
to the Files panel (left) first.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:
    print("Installing Whisper, about a minute...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("facilitator_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("facilitator_files.zip").extractall(".")
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)
PROTO or sys.exit("Upload facilitator_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg:", "found" if shutil.which("ffmpeg") else "NOT FOUND")

# GPU or not decides the model size: medium.en on a T4 is fast; on CPU it's a 3-minute press
# (measured), so small.en is used instead there -- same minutes, about 1 minute instead.
import torch
has_gpu = torch.cuda.is_available()
print(f"GPU: {'yes' if has_gpu else 'no'} -- expect the button press to take "
      f"{'under a minute' if has_gpu else 'about 1 minute (small.en, no GPU here)'}")
os.environ["WHISPER_MODEL"] = "medium.en" if has_gpu else "small.en"

# PRELOAD now, not during the hook -- the first call to Whisper downloads the model and loads
# it, which is slow; do that here, ahead of time.
import whisper
import transcriber
transcriber._model = whisper.load_model(transcriber.WHISPER_MODEL)
print(f"Whisper ({transcriber.WHISPER_MODEL}) preloaded and ready.")

# Both cell 3 and cell 4 need these -- imported here once, so either cell works
# on its own (cell 4 used to NameError if cell 3 never ran -- WP1 review finding 1).
from demo import run_demo
from IPython.display import Audio, HTML, Markdown, display


def big_minutes(minutes_markdown):
    """generate_minutes()'s fixed format (# title, **Attendees:**, ## headings,
    - / - [ ] bullets), as large HTML for a projector -- no code change to
    meeting_minutes.py (frozen), just a bigger rendering of its output here."""
    import html as html_lib
    lines_out = ['<div style="font-size:160%; line-height:1.5">']
    for line in minutes_markdown.splitlines():
        text = html_lib.escape(line)
        if line.startswith("# "):
            lines_out.append(f"<h1>{text[2:]}</h1>")
        elif line.startswith("## "):
            lines_out.append(f"<h2>{text[3:]}</h2>")
        elif line.startswith("- [ ] "):
            lines_out.append(f"<div>&#9744; {text[6:]}</div>")
        elif line.startswith("- "):
            lines_out.append(f"<div>&bull; {text[2:]}</div>")
        elif line.startswith("**") and line.endswith("**") is False and "**" in line:
            bold_text = text.replace("**", "<b>", 1).replace("**", "</b>", 1)
            lines_out.append(f"<p>{bold_text}</p>")
        elif line.strip():
            lines_out.append(f"<p>{text}</p>")
    lines_out.append("</div>")
    return HTML(chr(10).join(lines_out))


## 2. Record the skit from the browser microphone
Click record, do the skit (two presenters, `demo/script.txt`). Say each turn in one go, with no
long pause inside it, and leave a clear ~2-second pause before the other person speaks -- see the
pacing note above. Saves to `skit.wav`. Needs Colab (uses `google.colab.output`); on a laptop,
record with any app instead and skip to cell 3 with that file's path.

In [ ]:
from google.colab import output
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, display
import subprocess

RECORD_JS = """
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var recorder, chunks = [], micStream;
async function startRecording() {
  micStream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(micStream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
}
async function stopRecording() {
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  micStream.getTracks().forEach(t => t.stop());  // release the mic / turn off the browser's recording indicator
  const blob = new Blob(chunks);
  return await b2text(blob);
}
"""

def record_skit(out_path: str = "skit.wav") -> str:
    """Browser mic -> out_path (WAV). Click to start, run the next cell's
    input() prompt (or just wait for it) to stop -- two calls, not a
    widget, since that is more reliable across Colab versions."""
    display(Javascript(RECORD_JS))
    eval_js("startRecording()")
    input("Recording... press Enter here when the skit is done. ")
    data_url = eval_js("stopRecording()")
    _, encoded = data_url.split(",", 1)
    webm_path = out_path.replace(".wav", ".webm")
    with open(webm_path, "wb") as f:
        f.write(b64decode(encoded))
    # ffmpeg converts the browser's webm/opus to WAV -- Whisper reads WAV directly.
    result = subprocess.run(["ffmpeg", "-y", "-i", webm_path, out_path], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)   # show WHY ffmpeg failed, not just that it did
        raise RuntimeError(f"ffmpeg failed converting {webm_path} -> {out_path} (see stderr above)")
    print(f"Saved {out_path}")
    return out_path

skit_path = record_skit()


## 3. Press. One. Button. (the live recording)

In [ ]:
minutes = run_demo(skit_path, banner=False)
display(big_minutes(minutes))


## 4. Backup: if the recording didn't work, use the two-voice backup WAV instead
`demo/backup_meeting.wav` -- see `demo/make_backup_wav.py` for how it was made, and
`demo/FACILITATOR.md` for when to use it.

In [ ]:
backup_path = next(p for p in ("demo/backup_meeting.wav", "../demo/backup_meeting.wav") if os.path.isfile(p))
display(Audio(backup_path, autoplay=False))   # play this to the room -- it was never played before this fix
minutes = run_demo(backup_path, banner=False)
display(big_minutes(minutes))


## 5. If Whisper won't load at all: text-only fallback (no audio needed)
Instant, no model: runs `demo/script.txt` as typed text through the same minutes pipeline.
Not "press one button" on the actual skit, but keeps the hook going if the model truly
won't load.


In [ ]:
minutes = run_demo(banner=False)   # no argument -- demo/script.txt, no Whisper at all
display(big_minutes(minutes))
